# 02 Stretch: a Foundry IQ knowledge graph with your ontology

In challenge 4 you built a knowledge base over the Strandvik documents in the Foundry portal. A **knowledge graph** (preview) goes one step further: while it reads the documents, a model extracts **entities and relationships** that follow an **ontology** (your units, reservations, owners, customers, contracts...). Questions like "who must approve moving SD9-R-0733?" can then follow the graph.

Preview facts that matter:
- REST only, Search API version `2026-11-01-preview`.
- `graphExtraction` needs a chat model (`chatCompletionModel`): we use `gpt-5.6-luna`. The chat model is only allowed together with `graphExtraction`.
- The ontology is fixed when the knowledge source is created. A new ontology version needs a **new** knowledge source, and the graph needs its **own** knowledge base.
- Upload the documents first (challenge 1, or notebook 00).

In [ ]:
# Business role: start from data/foundry-iq/strandvik-ontology.jsonld, copy it, change it.
ONTOLOGY_FILE = "strandvik-ontology.jsonld"
VERSION = "v1"   # bump this (v2, v3...) for every new ontology: a new source and KB are created

In [ ]:
# Run once if the packages are missing (the Codespace installs them for you).
# Packages come through the Microsoft package proxy.
# %pip install -q -r requirements.txt --index-url https://packagefeedproxy.microsoft.io/pypi/simple/

In [ ]:
# Settings come from ../.env, written by create-env.sh (or create-env.ps1).
# You sign in once with "az login"; the notebook reuses that sign-in.
import os, json, time
from pathlib import Path
import requests
from dotenv import load_dotenv
from azure.identity import AzureCliCredential, ChainedTokenCredential, DefaultAzureCredential

HACK_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(HACK_DIR / ".env", override=True)
env = os.environ.get

credential = ChainedTokenCredential(AzureCliCredential(process_timeout=30),
                                    DefaultAzureCredential(exclude_cli_credential=True))

def call(method, url, scope, body=None, ok=(200, 201, 202, 204), headers=None, data=None):
    """One REST call with an Entra token. Retries throttling. Raises on any other error."""
    hdrs = {"Authorization": "Bearer " + credential.get_token(scope).token}
    if body is not None:
        hdrs["Content-Type"] = "application/json"
    hdrs.update(headers or {})
    for attempt in range(8):
        r = requests.request(method, url, headers=hdrs, json=body, data=data, timeout=180)
        # 430 = the Fabric capacity is busy (small F2 capacity), 429/5xx = throttling.
        if r.status_code in (429, 430, 500, 502, 503) and attempt < 7:
            time.sleep(int(r.headers.get("Retry-After", 10 + 10 * attempt)))
            continue
        break
    if r.status_code not in ok:
        raise RuntimeError(f"{method} {url.split('?')[0]} -> HTTP {r.status_code}: {r.text[:1000]}")
    return r

In [ ]:
SEARCH = env("SEARCH_ENDPOINT").rstrip("/")
SEARCH_SCOPE = "https://search.azure.com/.default"
PREVIEW = "2026-11-01-preview"
ACCOUNT = env("FOUNDRY_ACCOUNT")
STORAGE_ID = (f"/subscriptions/{env('AZURE_SUBSCRIPTION_ID')}/resourceGroups/{env('AZURE_RESOURCE_GROUP')}"
              f"/providers/Microsoft.Storage/storageAccounts/{env('STORAGE_ACCOUNT')}")
SOURCE, KB = f"strandvik-graph-{VERSION}", f"strandvik-graph-{VERSION}-kb"

def search(method, path, body=None, ok=(200, 201, 204)):
    r = call(method, f"{SEARCH}/{path}?api-version={PREVIEW}", SEARCH_SCOPE, body, ok)
    return r.json() if r.content else {}

def model(deployment):
    # The Search service calls your models with its own identity (granted by the lab).
    return {"kind": "azureOpenAI", "azureOpenAIParameters": {
        "resourceUri": f"https://{ACCOUNT}.openai.azure.com", "deploymentId": deployment, "modelName": deployment}}

CHAT = model(env("JUDGE_MODEL_DEPLOYMENT_NAME", "gpt-5.6-luna"))
EMBED = model(env("EMBEDDING_DEPLOYMENT_NAME", "text-embedding-3-large"))
ontology = json.loads((HACK_DIR / "data" / "foundry-iq" / ONTOLOGY_FILE).read_text(encoding="utf-8"))
print("ontology loaded:", ONTOLOGY_FILE)

## 1. Knowledge source with graph extraction (takes a few minutes)

In [ ]:
source = {
    "name": SOURCE, "kind": "azureBlob",
    "description": "Strandvik policies, contracts, logistics rules and product sheet, as a knowledge graph.",
    "azureBlobParameters": {
        "connectionString": f"ResourceId={STORAGE_ID};",
        "containerName": env("DOCUMENTS_CONTAINER", "documents"),
        "ingestionParameters": {
            "contentExtractionMode": "minimal", "disableImageVerbalization": True,
            "embeddingModel": EMBED,
            "chatCompletionModel": CHAT,          # allowed only with graphExtraction
        },
        "graphExtraction": {
            "graphName": SOURCE, "pageDegreeOfParallelism": 2,
            "ontology": {"kind": "inline", "name": Path(ONTOLOGY_FILE).stem,
                         "value": json.dumps(ontology, separators=(",", ":"))},
        },
    },
    "resultsProcessing": "none",
}
search("PUT", f"knowledgesources('{SOURCE}')", source)

# Wait until the first ingestion run has finished.
for _ in range(120):
    s = search("GET", f"knowledgesources('{SOURCE}')/status")
    last = s.get("lastSynchronizationState") or {}
    if last.get("endTime") and not s.get("currentSynchronizationState"):
        break
    time.sleep(10)
print(json.dumps(last, indent=2)[:800])
assert not last.get("itemsUpdatesFailed"), "Some documents failed, see the errors above."
time.sleep(15)  # graph writes finish shortly after the indexer run

## 2. Knowledge base on the graph, then ask it

In [ ]:
search("PUT", f"knowledgebases('{KB}')", {
    "name": KB, "knowledgeSources": [{"name": SOURCE}], "models": [CHAT],
    "retrievalReasoningEffort": {"kind": "low"}, "outputMode": "extractiveData",
    "retrievalInstructions": "Use graph_search for questions about units, reservations, owners, customers, "
                             "contracts and approvals. Traverse the graph rather than using text search."})

QUESTION = "The Rotterdam SD-900 is reserved. Who must approve moving it to Kolvo, and in what form?"
answer = search("POST", f"knowledgebases('{KB}')/retrieve", {
    "messages": [{"role": "user", "content": [{"type": "text", "text": QUESTION}]}],
    "includeActivity": True, "maxRuntimeInSeconds": 90,
    "knowledgeSourceParams": [{"kind": "azureBlob", "knowledgeSourceName": SOURCE, "includeReferences": True,
                               "graphParameters": {"includeQuery": True}}]})
for item in answer.get("response", []):
    for c in item.get("content", []):
        print(c.get("text", "")[:2000])
print("\nactivity:", [a.get("type") for a in answer.get("activity", [])])

## 3. Use it in an agent

The new knowledge base appears in the Foundry portal (Build > Knowledge) next to the one you made in challenge 4. Open it and choose **Use in an agent**, or add it to the Service agent. Compare the answers with and without the graph.

Changed the ontology? Set `VERSION = "v2"` and run again: a new source and knowledge base are created; the old ones stay for comparison.